# DurakZero Rule-Based Benchmark Evaluation

Simulate DurakZero checkpoints against reference heuristic opponents to estimate how often the learned policy defeats common rule-based agents.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace
from typing import Callable, Dict, Iterable, Optional, Tuple

import numpy as np
import torch

from douzero.env.env import (
    ACTION_END_ATTACK,
    ACTION_TAKE_CARDS,
    MAX_ATTACK_CARDS,
    NUM_CARDS,
    DurakState,
    apply_action,
    build_observation,
    card_rank,
    card_suit,
    current_player,
    initial_state,
    legal_actions,
)
from douzero.evaluation.simulation import load_model


In [ ]:
NUM_SUITS = 4
NUM_RANKS = NUM_CARDS // NUM_SUITS

PolicyFn = Callable[[DurakState, int, np.random.Generator], int]

def legal_indices(state: DurakState) -> np.ndarray:
    mask = legal_actions(state)
    return np.flatnonzero(mask)

def ranks_on_table(state: DurakState) -> set[int]:
    ranks = {card_rank(atk) for atk, _ in state.table}
    ranks.update(card_rank(defn) for _, defn in state.table if defn is not None)
    return ranks

def can_cover(defense: int, attack: int, trump_suit: int) -> bool:
    suit = card_suit(defense)
    atk_suit = card_suit(attack)
    if suit == atk_suit:
        return card_rank(defense) > card_rank(attack)
    if suit == trump_suit and atk_suit != trump_suit:
        return True
    return False

def unseen_counts_by_suit(state: DurakState) -> Dict[int, int]:
    counts = {suit: NUM_RANKS for suit in range(NUM_SUITS)}
    for card in state.seen_cards:
        suit = card_suit(card)
        counts[suit] = max(0, counts[suit] - 1)
    return counts

def greedy_policy(state: DurakState, player: int, rng: np.random.Generator | None = None) -> int:
    if current_player(state) != player:
        raise ValueError('Policy called for non-active player')
    legal = legal_indices(state)
    if legal.size == 0:
        raise ValueError('No legal actions available')
    cards = [a for a in legal if a < NUM_CARDS]
    if player == state.attacker:
        if not state.table:
            non_trump = [c for c in cards if card_suit(c) != state.trump_suit]
            if non_trump:
                return min(non_trump, key=lambda c: (card_rank(c), card_suit(c)))
            if cards:
                return min(cards, key=lambda c: (card_suit(c) == state.trump_suit, card_rank(c)))
        else:
            ranks = ranks_on_table(state)
            matching = [c for c in cards if card_rank(c) in ranks]
            if matching:
                non_trump = [c for c in matching if card_suit(c) != state.trump_suit]
                if non_trump:
                    return min(non_trump, key=lambda c: card_rank(c))
                return min(matching, key=lambda c: (card_suit(c) == state.trump_suit, card_rank(c)))
            if ACTION_END_ATTACK in legal:
                return ACTION_END_ATTACK
        if ACTION_END_ATTACK in legal and not cards:
            return ACTION_END_ATTACK
        return int(legal[0])
    else:
        if cards:
            uncovered = [atk for atk, defense in state.table if defense is None]
            if uncovered:
                target = uncovered[0]
                candidates = [c for c in cards if can_cover(c, target, state.trump_suit)]
                if candidates:
                    non_trump = [c for c in candidates if card_suit(c) != state.trump_suit]
                    if non_trump:
                        return min(non_trump, key=lambda c: card_rank(c))
                    return min(candidates, key=lambda c: card_rank(c))
        if ACTION_TAKE_CARDS in legal:
            return ACTION_TAKE_CARDS
        return int(legal[0])

def difficult_policy(state: DurakState, player: int, rng: np.random.Generator | None = None) -> int:
    if current_player(state) != player:
        raise ValueError('Policy called for non-active player')
    legal = legal_indices(state)
    if legal.size == 0:
        raise ValueError('No legal actions available')
    cards = [a for a in legal if a < NUM_CARDS]
    unseen_counts = unseen_counts_by_suit(state)
    if player == state.attacker:
        if not state.table:
            if cards:
                non_trump = [c for c in cards if card_suit(c) != state.trump_suit]
                pool = non_trump if non_trump else cards
                return min(
                    pool,
                    key=lambda c: (
                        unseen_counts.get(card_suit(c), NUM_RANKS),
                        card_suit(c) == state.trump_suit,
                        card_rank(c),
                    ),
                )
        else:
            ranks = ranks_on_table(state)
            matching = [c for c in cards if card_rank(c) in ranks]
            if state.defender_taking and matching:
                return min(
                    matching,
                    key=lambda c: (
                        card_suit(c) == state.trump_suit,
                        unseen_counts.get(card_suit(c), NUM_RANKS),
                        card_rank(c),
                    ),
                )
            if matching:
                non_trump = [c for c in matching if card_suit(c) != state.trump_suit]
                pool = non_trump if non_trump else matching
                return min(
                    pool,
                    key=lambda c: (
                        unseen_counts.get(card_suit(c), NUM_RANKS),
                        card_suit(c) == state.trump_suit,
                        card_rank(c),
                    ),
                )
            if ACTION_END_ATTACK in legal:
                return ACTION_END_ATTACK
        if cards:
            return min(cards, key=lambda c: (card_suit(c) == state.trump_suit, card_rank(c)))
        if ACTION_END_ATTACK in legal:
            return ACTION_END_ATTACK
        return int(legal[0])
    else:
        if cards:
            uncovered = [atk for atk, defense in state.table if defense is None]
            if uncovered:
                target = uncovered[0]
                candidates = [c for c in cards if can_cover(c, target, state.trump_suit)]
                if candidates:
                    non_trump = [c for c in candidates if card_suit(c) != state.trump_suit]
                    pool = non_trump if non_trump else candidates
                    return min(pool, key=lambda c: (card_suit(c) == state.trump_suit, card_rank(c)))
        if ACTION_TAKE_CARDS in legal:
            return ACTION_TAKE_CARDS
        return int(legal[0])


In [ ]:
@dataclass
class MatchupResult:
    model_name: str
    opponent_name: str
    games: int
    model_wins: int
    opponent_wins: int
    win_rate: float
    seat0_wins: int
    seat1_wins: int

def play_game(model, opponent_policy: PolicyFn, model_seat: int, rng: np.random.Generator, device: torch.device) -> Optional[int]:
    state = initial_state(rng)
    flags = SimpleNamespace(exp_epsilon=0.0)
    while not state.terminal:
        active = current_player(state)
        if active == model_seat:
            obs = build_observation(state, player=active)
            state_tensor = torch.from_numpy(obs['state']).to(device).unsqueeze(0)
            action_embeddings = torch.from_numpy(obs['action_embeddings']).to(device)
            with torch.no_grad():
                outputs = model.act(obs['position'], state_tensor.squeeze(0), action_embeddings, flags=flags)
            action_idx = int(outputs['action_index'])
            action = int(obs['legal_actions'][action_idx])
        else:
            state_copy = state.copy()
            action = opponent_policy(state_copy, active, rng)
        state, done, winner = apply_action(state, action)
        if done:
            return winner
    return state.winner

def evaluate_against(model, model_name: str, opponent_name: str, opponent_policy: PolicyFn, num_games: int = 100, seed: int = 0, device: torch.device | None = None) -> MatchupResult:
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    rng = np.random.default_rng(seed)
    model.eval()
    model_wins = 0
    seat_wins = {0: 0, 1: 0}
    for game_idx in range(num_games):
        model_seat = game_idx % 2
        winner = play_game(model, opponent_policy, model_seat, rng, device)
        if winner is None:
            continue
        if winner == model_seat:
            model_wins += 1
            seat_wins[model_seat] += 1
    opponent_wins = num_games - model_wins
    win_rate = model_wins / num_games if num_games else 0.0
    return MatchupResult(
        model_name=model_name,
        opponent_name=opponent_name,
        games=num_games,
        model_wins=model_wins,
        opponent_wins=opponent_wins,
        win_rate=win_rate,
        seat0_wins=seat_wins[0],
        seat1_wins=seat_wins[1],
    )


In [ ]:
def load_available_models(root: Path) -> Dict[str, Path]:
    checkpoints: Dict[str, Path] = {}
    for subdir in ['continuing_run', 'durakzero_notebook', 'vastai_run']:
        path = root / subdir / 'model.tar'
        if path.exists():
            checkpoints[subdir] = path
    return checkpoints

def summarize_matchups(results: Iterable[MatchupResult]):
    try:
        import pandas as pd  # type: ignore
    except Exception:
        pd = None
    rows = [
        {
            'Model': r.model_name,
            'Opponent': r.opponent_name,
            'Games': r.games,
            'Model wins': r.model_wins,
            'Opponent wins': r.opponent_wins,
            'Win rate': f"{r.win_rate:.3f}",
            'Seat0 wins': r.seat0_wins,
            'Seat1 wins': r.seat1_wins,
        }
        for r in results
    ]
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        for row in rows:
            print(row)


In [ ]:
CHECKPOINT_ROOT = Path('durakzero_checkpoints')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_GAMES = 100
SEED = 123

checkpoints = load_available_models(CHECKPOINT_ROOT)
if not checkpoints:
    raise FileNotFoundError(f'No model.tar checkpoints found under {CHECKPOINT_ROOT.resolve()}')

opponents: Dict[str, PolicyFn] = {
    'Greedy baseline': greedy_policy,
    'Difficult rule-based': difficult_policy,
}

results: list[MatchupResult] = []
for name, path in checkpoints.items():
    model = load_model(str(path), device=DEVICE)
    for opponent_name, policy in opponents.items():
        result = evaluate_against(
            model,
            model_name=name,
            opponent_name=opponent_name,
            opponent_policy=policy,
            num_games=NUM_GAMES,
            seed=SEED,
            device=DEVICE,
        )
        results.append(result)
summarize_matchups(results)
